# Phase 0 — Reproducibility and local baseline

Canonical end-to-end smoke notebook for a clean experimental machine. It freezes inputs, builds the CodeQL/IRIS containers, runs one CodeQL case, runs the legacy IRIS compatibility case, verifies the primary Qwen 3B Q4_K_M GGUF, and writes the phase gate report.

> Primary research model: `Qwen/Qwen2.5-Coder-3B-Instruct-GGUF`, `Q4_K_M`, local `llama.cpp`, CUDA 12.4. The legacy 1.5B model is executed only to preserve the Phase 0 IRIS-v1 comparison contract.


In [ ]:
from __future__ import annotations

import os
import sys
from datetime import UTC, datetime
from pathlib import Path


def locate_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "workflows").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the MaxVera repository")


ROOT = locate_root()
sys.path.insert(0, str(ROOT / "workflows" / "scripts" / "common"))
from notebook_helpers import (  # noqa: E402
    artifact_links,
    command_probe,
    json_dashboard,
    run_step,
    status_table,
)

PROFILE = os.getenv("MAXVERA_PROFILE", "smoke")
EXECUTE = os.getenv("MAXVERA_EXECUTE", "1") == "1"
RUN_LEGACY_IRIS = os.getenv("MAXVERA_RUN_LEGACY_IRIS", "1") == "1"
RUN_PRIMARY_MODEL_SMOKE = os.getenv("MAXVERA_RUN_MODEL_SMOKE", "1") == "1"
stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
CODEQL_RUN_ID = f"phase0_codeql_smoke_{stamp}"
IRIS_RUN_ID = f"phase0_iris_legacy_qwen15b_{stamp}"
MODEL_SMOKE_ID = f"phase0_qwen3b_q4km_{stamp}"
MODEL_PATH = ROOT / "models" / "qwen2.5-coder-3b-instruct-q4_k_m.gguf"
MODEL_SMOKE_RESULT = (
    ROOT / "results" / "phases" / "phase_00" / "model_smoke" / f"{MODEL_SMOKE_ID}.json"
)
print({"root": str(ROOT), "profile": PROFILE, "execute": EXECUTE, "stamp": stamp})

## 1. Preflight

The table must show Docker, NVIDIA and the pinned llama.cpp server before expensive cells start. For Quadro P2000, use the CUDA 12.4 build; do not use CUDA 13.


In [ ]:
preflight = [
    command_probe("Git", ["git", "--version"]),
    command_probe("uv", ["uv", "--version"]),
    command_probe("Docker", ["docker", "version", "--format", "{{.Server.Version}}"]),
    command_probe(
        "NVIDIA driver",
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    ),
    command_probe("llama.cpp b10590", ["llama-server", "--version"]),
]
status_table(preflight)
if EXECUTE and any(status == "FAIL" for _, status, _ in preflight):
    raise RuntimeError("Preflight failed; fix the missing runtime before continuing")

## 2. Environment, frozen data and pinned downloads

The Qwen cell downloads only the exact immutable revision and refuses a mismatched SHA-256. A verified cached model is reused without network I/O.


In [ ]:
setup_steps = [
    (
        "Sync Python environment",
        ["uv", "sync", "--extra", "dev", "--extra", "notebook", "--locked"],
    ),
    (
        "Bootstrap pinned upstream repositories",
        ["uv", "run", "python", "scripts/data/bootstrap_upstream.py"],
    ),
    ("Prepare frozen Phase 0 datasets", ["uv", "run", "python", "scripts/data/prepare_phase0.py"]),
    ("Fetch pinned CodeQL archive", ["uv", "run", "python", "scripts/toolchain/fetch_codeql.py"]),
    (
        "Fetch pinned Qwen 3B Q4_K_M GGUF",
        ["uv", "run", "python", "scripts/toolchain/fetch_qwen_gguf.py"],
    ),
]
setup_results = [
    run_step(title, argv, root=ROOT, execute=EXECUTE, timeout_seconds=3600)
    for title, argv in setup_steps
]
artifact_links(
    [
        "data/manifests/datasets",
        "data/runtime_views",
        "tools/downloads",
        MODEL_PATH,
    ],
    root=ROOT,
)

## 3. Build pinned containers and validate the harness


In [ ]:
container_steps = [
    (
        "Build JDK 17 image",
        [
            "docker",
            "build",
            "--platform",
            "linux/amd64",
            "-t",
            "maxvera-phase0:jdk17",
            "docker/phase0",
        ],
    ),
    (
        "Build CodeQL image",
        [
            "docker",
            "build",
            "--platform",
            "linux/amd64",
            "-f",
            "docker/phase0/Dockerfile.codeql",
            "-t",
            "maxvera-phase0:codeql",
            "tools/downloads",
        ],
    ),
    (
        "Build legacy IRIS image",
        [
            "docker",
            "build",
            "--platform",
            "linux/amd64",
            "-f",
            "docker/phase0/Dockerfile.iris",
            "-t",
            "maxvera-phase0:iris-local",
            "docker/phase0",
        ],
    ),
    (
        "Strict toolchain audit",
        ["uv", "run", "openiris-doctor", "--config", "configs/toolchain/phase0.yaml", "--strict"],
    ),
    ("Test suite", ["uv", "run", "pytest"]),
]
container_results = [
    run_step(title, argv, root=ROOT, execute=EXECUTE, timeout_seconds=3600)
    for title, argv in container_steps
]
artifact_links(["results/phases/phase_00/audits", "tests"], root=ROOT)

## 4. CodeQL smoke baseline

This creates one append-only run and one aggregate JSONL record. Re-running the notebook creates a new timestamped ID.


In [ ]:
run_step(
    "CodeQL smoke",
    [
        "uv",
        "run",
        "python",
        "scripts/baselines/run_codeql_docker.py",
        "--config",
        "configs/experiments/phase0_codeql_smoke.yaml",
        "--run-id",
        CODEQL_RUN_ID,
    ],
    root=ROOT,
    execute=EXECUTE,
    timeout_seconds=7200,
)
CODEQL_RESULT = (
    ROOT / "results" / "phases" / "phase_00" / "baselines" / "codeql" / f"{CODEQL_RUN_ID}.jsonl"
)
CODEQL_DATABASE = ROOT / "runs" / CODEQL_RUN_ID / "workspace" / "database"
artifact_links([CODEQL_RESULT, CODEQL_DATABASE], root=ROOT)

## 5. Legacy IRIS-v1 compatibility smoke

This is a historical reproduction exception: it uses the pinned 1.5B Transformers path and an oracle CWE interface. It is not the primary model for Phase 1 onward.


In [ ]:
INSTANCE_ID = "oi_d67309725ca108a035d5bb04a9b302b4"
legacy_iris_command = [
    "uv",
    "run",
    "python",
    "scripts/baselines/run_iris_docker.py",
    "--config",
    "configs/experiments/phase0_iris_local_smoke.yaml",
    "--instance-id",
    INSTANCE_ID,
    "--runtime-manifest",
    "data/runtime_views/cwe_bench_java_legacy_698fb724/manifest.jsonl",
    "--runtime-repository",
    f"data/runtime_views/cwe_bench_java_legacy_698fb724/{INSTANCE_ID}/repository",
    "--ground-truth",
    "data/ground_truth/cwe_bench_java_legacy_698fb724/ground_truth.jsonl",
    "--codeql-database",
    str(CODEQL_DATABASE),
    "--image",
    "maxvera-phase0:iris-local",
    "--run-id",
    IRIS_RUN_ID,
]
run_step(
    "Legacy IRIS smoke",
    legacy_iris_command,
    root=ROOT,
    execute=EXECUTE and RUN_LEGACY_IRIS,
    timeout_seconds=7200,
)
IRIS_RESULT = (
    ROOT / "results" / "phases" / "phase_00" / "baselines" / "iris" / f"{IRIS_RUN_ID}.jsonl"
)
artifact_links([IRIS_RESULT, ROOT / "runs" / IRIS_RUN_ID / "artifacts" / "iris"], root=ROOT)

## 6. Primary local Qwen 3B Q4_K_M CUDA smoke

The script starts the pinned `llama-server`, performs one OpenAI-compatible chat request, samples GPU memory, retains the server log, and stops the server. CUDA must appear in the log.


In [ ]:
run_step(
    "Primary Qwen 3B local CUDA smoke",
    [
        "uv",
        "run",
        "python",
        "workflows/scripts/phase_00/smoke_local_qwen.py",
        "--model",
        str(MODEL_PATH),
        "--output",
        str(MODEL_SMOKE_RESULT),
    ],
    root=ROOT,
    execute=EXECUTE and RUN_PRIMARY_MODEL_SMOKE,
    timeout_seconds=1200,
)
artifact_links([MODEL_SMOKE_RESULT, MODEL_SMOKE_RESULT.with_suffix(".server.log")], root=ROOT)
if MODEL_SMOKE_RESULT.exists():
    json_dashboard(MODEL_SMOKE_RESULT, ["status", "runtime_seconds"])

## 7. Final gate and result links


In [ ]:
run_step(
    "Generate Phase 0 report",
    ["uv", "run", "python", "scripts/phase0.py"],
    root=ROOT,
    execute=EXECUTE,
)
STATUS_PATH = ROOT / "results" / "phases" / "phase_00" / "status.json"
REPORT_PATH = ROOT / "results" / "phases" / "phase_00" / "report.md"
if STATUS_PATH.exists():
    phase_status = json_dashboard(STATUS_PATH, ["status", "generated_at"])
    checks = [
        (name, "PASS" if passed else "WARN", str(passed))
        for name, passed in phase_status.get("checks", {}).items()
    ]
    status_table(checks)
artifact_links(
    [STATUS_PATH, REPORT_PATH, CODEQL_RESULT, IRIS_RESULT, MODEL_SMOKE_RESULT], root=ROOT
)
if EXECUTE and phase_status.get("status") not in {"SMOKE_PASS", "PASS"}:
    raise RuntimeError(f"Phase 0 gate did not pass: {phase_status.get('status')}")